# LeWM + CLAW on Google Colab
**Before starting:** (1) Runtime > Change runtime type > GPU. (2) Push your latest code to GitHub: the clone cell only gets committed code (`model=lewm_claw`, `eval_retrieval.py`, `latent_planner.py`, `tests/` must all be pushed).

**Runtime > Run all** runs the whole CLAW pipeline with the settings in the *Settings* cell: setup, data download, smoke tests, CLAW training, the z -> action retrieval check, and PushT planning with CEM over latent actions. The Google Drive cell asks for permission once. The 100-epoch baseline LeWM run is off by default (`RUN_BASELINE = False`).

In [ ]:
!nvidia-smi
import torch
print("torch", torch.__version__, "| cuda:", torch.cuda.is_available())
assert torch.cuda.is_available(), "No GPU - switch the runtime type to GPU first"
!df -h /content | tail -1

## 1. Clone the repo

In [ ]:
%cd /content
!git clone https://github.com/arshagarwal/le-wm.git
%cd /content/le-wm
!git log --oneline | head -3

## Settings
Edit these, then Runtime > Run all. Defaults are a short dev run (a few minutes per epoch at 112 px), not a full training run. For a real run set `CLAW_EPOCHS=100`, `CLAW_TRAIN_BATCHES=None`, `IMG_SIZE=224` (and lower `CLAW_BATCH` if you hit out-of-memory). `delta` (the adversary weight) ramps up over `CLAW_EPOCHS`, so keep it equal to the real run length.

In [ ]:
RUN_BASELINE = False      # True: also train plain LeWM for 100 epochs (very long on a T4)
CLAW_RUN = "claw_dev"     # name of the CLAW run: checkpoints/<CLAW_RUN>/weights_epoch_N.pt
CLAW_EPOCHS = 3
CLAW_TRAIN_BATCHES = 100  # batches per epoch; None = full epochs
CLAW_VAL_BATCHES = 20
IMG_SIZE = 112            # training image size; the retrieval and PushT eval cells reuse it
CLAW_BATCH = 32
CKPT_EVERY = 500          # save a resumable checkpoint to Drive every N steps (0 = epoch ends only)
RETR_REF, RETR_QUERY = 4096, 1024   # lookup-set / held-out windows for the retrieval check
EVAL_EPISODES = 10        # PushT episodes for the planning eval (paper-style eval uses 50)
CLAW_LIMITS = "" if CLAW_TRAIN_BATCHES is None else f"+trainer.limit_train_batches={CLAW_TRAIN_BATCHES} +trainer.limit_val_batches={CLAW_VAL_BATCHES}"

## 2. Install dependencies
This takes a few minutes. If pip reports a torch version change, that is fine.

In [ ]:
!pip install -q "stable-worldmodel[train]" hdf5plugin h5py hf_transfer pytest pygame pymunk shapely opencv-python-headless
!apt-get -qq install -y zstd > /dev/null
import stable_worldmodel, stable_pretraining, hdf5plugin
import pygame, pymunk, shapely, cv2   # PushT simulator deps (not part of the [train] extra)
from stable_worldmodel.data.format import list_formats
print("formats:", list_formats())
assert "hdf5" in list_formats(), "hdf5 format not registered - check hdf5plugin/h5py install"
print("deps OK")

## 3. Download and decompress the PushT dataset
The dataset is an `.h5.zst` archive (~13 GB). It goes under `$STABLEWM_HOME/datasets/`, which is where this version of stable-worldmodel looks.

In [ ]:
import os
os.environ["STABLEWM_HOME"] = "/content/stablewm"
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
DATA_DIR = "/content/stablewm/datasets"
os.makedirs(DATA_DIR, exist_ok=True)

!hf download quentinll/lewm-pusht --repo-type dataset --local-dir /content/dl_pusht
!ls -lh /content/dl_pusht

In [ ]:
# decompress; handles either a plain zstd'd .h5 or a zstd'd tar
!zstd -d -f /content/dl_pusht/pusht_expert_train.h5.zst -o /content/dl_pusht/out.bin
with open("/content/dl_pusht/out.bin", "rb") as f:
    magic = f.read(8)
if magic.startswith(b"\x89HDF"):
    !mv /content/dl_pusht/out.bin {DATA_DIR}/pusht_expert_train.h5
else:
    !tar -xf /content/dl_pusht/out.bin -C {DATA_DIR} && rm /content/dl_pusht/out.bin
!rm -f /content/dl_pusht/pusht_expert_train.h5.zst
!ls -lh {DATA_DIR}
!df -h /content | tail -1

In [ ]:
# sanity check: dataset loads and shows its columns
import stable_worldmodel as swm
ds = swm.data.load_dataset("pusht_expert_train.h5", frameskip=5, num_steps=4,
                           keys_to_load=["pixels", "action", "proprio", "state"],
                           keys_to_cache=["action", "proprio", "state"])
print(len(ds), "samples")
print({k: tuple(getattr(v, "shape", ())) for k, v in ds[0].items()})

## 4. Smoke test (about 1 minute)
Runs 5 train batches and 2 val batches to confirm everything works before the full run.
bf16 is used on GPUs that support it (A100/L4); T4 falls back to fp16 mixed precision.

In [ ]:
import torch
PRECISION = "bf16" if torch.cuda.is_bf16_supported() else "16-mixed"
print("precision:", PRECISION)

COMMON = (
    "data=pusht data.dataset.name=pusht_expert_train.h5 "
    f"trainer.precision={PRECISION} num_workers=2 wandb.enabled=False"
)
!cd /content/le-wm && STABLEWM_HOME=/content/stablewm python train.py {COMMON} \
    trainer.max_epochs=1 +trainer.limit_train_batches=5 +trainer.limit_val_batches=2 \
    output_model_name=smoke subdir=smoke

## 4b. Unit tests (a few seconds)
Checks the CLAW modules, the lookup and the latent planner on random tensors.

In [ ]:
!cd /content/le-wm && python -m pytest tests -q

## 5. Training
Colab sessions can disconnect, so the next cell mounts Google Drive and writes checkpoints there. The baseline LeWM run (100 epochs, batch 128, lr 5e-5) only runs if `RUN_BASELINE = True`. If the session disconnects, re-run the Drive cell, the resume cell, and the training cell: training resumes from the last finished epoch (same `subdir`). If you hit an out-of-memory error, add `loader.batch_size=64`.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
os.makedirs("/content/drive/MyDrive/lewm", exist_ok=True)
os.environ["STABLEWM_HOME"] = "/content/drive/MyDrive/lewm"
# Lightning writes resumable ckpts under SPT_CACHE_DIR; keep them on Drive too
os.environ["SPT_CACHE_DIR"] = "/content/drive/MyDrive/lewm/spt_cache"
os.makedirs("/content/drive/MyDrive/lewm/spt_cache", exist_ok=True)
# dataset stays on local disk (fast); symlink it into the Drive-backed home
os.makedirs("/content/drive/MyDrive/lewm/datasets", exist_ok=True)
!ln -sf /content/stablewm/datasets/pusht_expert_train.h5 /content/drive/MyDrive/lewm/datasets/pusht_expert_train.h5
!ls -l /content/drive/MyDrive/lewm/datasets

In [ ]:
# Resume support (baseline run): train.py resumes from <STABLEWM_HOME>/checkpoints/<subdir>/lewm_weights.ckpt,
# but Lightning writes last.ckpt under SPT_CACHE_DIR. Copy the newest one into place (no-op on a fresh run).
import glob, shutil
if RUN_BASELINE:
    run_dir = "/content/drive/MyDrive/lewm/checkpoints/pusht_run1"
    os.makedirs(run_dir, exist_ok=True)
    cands = glob.glob("/content/drive/MyDrive/lewm/spt_cache/runs/*/*/*/checkpoints/last.ckpt")
    if cands:
        newest = max(cands, key=os.path.getmtime)
        shutil.copy(newest, f"{run_dir}/lewm_weights.ckpt")
        print("will resume from", newest)
    else:
        print("no previous run found - starting fresh")
else:
    print("RUN_BASELINE = False: skipping")

In [ ]:
if RUN_BASELINE:
    !cd /content/le-wm && STABLEWM_HOME=/content/drive/MyDrive/lewm python train.py {COMMON} \
        output_model_name=lewm subdir=pusht_run1
else:
    print("RUN_BASELINE = False: skipping the baseline LeWM run")

## 5b. CLAW latent-action run
Adds the CLAW latent action model + adversarial regularizer (`model=lewm_claw`). Extra metrics logged per step: `adv_loss`, `ground_loss`, `delta` (GRL scale, ramps 0 -> 0.999 over the run), `z_std` and `z_erank` (collapse check; both near 0 = collapsed), `adv_rel` (adversary error / baseline variance; near 1 = no leakage, near 0 = z leaks the next frame).
CLAW runs use their own Lightning cache dir (`spt_cache_claw`), so resuming a CLAW run never picks up the baseline's checkpoint (and vice versa). To resume after a disconnect, re-run the Drive cell and the training cell below.

In [ ]:
# CLAW smoke test (about 1 minute)
!cd /content/le-wm && SPT_CACHE_DIR=/content/spt_cache_smoke STABLEWM_HOME=/content/stablewm python train.py {COMMON} model=lewm_claw \
    trainer.max_epochs=1 +trainer.limit_train_batches=5 +trainer.limit_val_batches=2 \
    output_model_name=claw_smoke subdir=claw_smoke

In [ ]:
# CLAW training on Drive. Resume: train.py looks for checkpoints/<subdir>/<output_model_name>_weights.ckpt;
# copy the newest CLAW last.ckpt there.
import glob, shutil
CLAW_CACHE = "/content/drive/MyDrive/lewm/spt_cache_claw"
claw_dir = f"/content/drive/MyDrive/lewm/checkpoints/{CLAW_RUN}"
os.makedirs(CLAW_CACHE, exist_ok=True); os.makedirs(claw_dir, exist_ok=True)
# train.py also writes this file itself every CKPT_EVERY steps; only overwrite it with an epoch-end
# last.ckpt if that one is newer (never replace a newer step checkpoint with an older one).
target = f"{claw_dir}/{CLAW_RUN}_weights.ckpt"
cands = glob.glob(f"{CLAW_CACHE}/runs/*/*/*/checkpoints/last.ckpt")
if cands and (not os.path.exists(target) or os.path.getmtime(max(cands, key=os.path.getmtime)) > os.path.getmtime(target)):
    shutil.copy(max(cands, key=os.path.getmtime), target)
    print("will resume CLAW run from", max(cands, key=os.path.getmtime))
elif os.path.exists(target):
    print("will resume CLAW run from", target)
else:
    print("no previous CLAW run found - starting fresh")
!cd /content/le-wm && SPT_CACHE_DIR={CLAW_CACHE} STABLEWM_HOME=/content/drive/MyDrive/lewm python train.py {COMMON} model=lewm_claw \
    trainer.max_epochs={CLAW_EPOCHS} {CLAW_LIMITS} \
    img_size={IMG_SIZE} loader.batch_size={CLAW_BATCH} \
    output_model_name={CLAW_RUN} subdir={CLAW_RUN} ckpt_every_n_steps={CKPT_EVERY}

## 5c. CLAW evaluation (`eval_claw.py`)
1. **Retrieval check** (no simulator): maps held-out windows' z to real actions by nearest neighbour in a lookup set built from reference windows. `knn_rel`: 0 = recovers the action exactly, 1 = no better than predicting the mean action.
2. **PushT planning**: CEM over latent actions with the world model, each planned z mapped to a real action by the same lookup, executed in the simulator. Needs a GPU. Results and videos: `MyDrive/lewm/<CLAW_RUN>/`. Set `EVAL_EPISODES = 50` in Settings for a paper-style eval.

In [ ]:
CLAW_CKPT = f"{CLAW_RUN}/weights_epoch_{CLAW_EPOCHS}.pt"      # relative to $STABLEWM_HOME/checkpoints
os.environ["SDL_VIDEODRIVER"] = "dummy"                       # headless pygame rendering
EVAL = f"cd /content/le-wm && STABLEWM_HOME=/content/drive/MyDrive/lewm python eval_claw.py --config-name=pusht policy={CLAW_CKPT} eval.img_size={IMG_SIZE} +num_ref={RETR_REF} +num_workers=2"
!{EVAL} +mode=retrieval +num_query={RETR_QUERY}
!{EVAL} eval.num_eval={EVAL_EPISODES} output.filename=pusht_claw_results.txt
!tail -n 4 /content/drive/MyDrive/lewm/{CLAW_RUN}/pusht_claw_results.txt

## 6. Check outputs
CLAW weights: `MyDrive/lewm/checkpoints/<CLAW_RUN>/weights_epoch_N.pt`; planning results and videos: `MyDrive/lewm/<CLAW_RUN>/`.
Baseline weights (if run): `MyDrive/lewm/checkpoints/lewm/weights_epoch_N.pt`. Resumable Lightning checkpoints: `MyDrive/lewm/spt_cache*/runs/.../checkpoints/last.ckpt`.

In [ ]:
!ls -lh /content/drive/MyDrive/lewm/checkpoints/{CLAW_RUN}
!ls -lh /content/drive/MyDrive/lewm/{CLAW_RUN} 2>/dev/null | head
!ls -lh /content/drive/MyDrive/lewm/checkpoints/lewm 2>/dev/null | tail -3
!find /content/drive/MyDrive/lewm/spt_cache* -name '*.ckpt' | head

## Optional: weights & biases logging
Add `wandb.enabled=True wandb.config.entity=<your_entity> wandb.config.project=<your_project>` to the training command and run `!wandb login` first.